In [5]:


import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.gridspec import GridSpec
import warnings

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted")

# ── 1. LOAD DATA 
df = sns.load_dataset("titanic")
print("=" * 60)
print("TITANIC DATASET — EXPLORATORY DATA ANALYSIS")
print("=" * 60)

# ── 2. BASIC OVERVIEW ────────────────────────────────────────
print("\n📌 Shape:", df.shape)
print("\n📌 Column Names:\n", df.columns.tolist())
print("\n📌 Data Types:\n", df.dtypes)
print("\n📌 First 5 Rows:\n", df.head())

# ── 3. STATISTICAL SUMMARY ───────────────────────────────────
print("\n📊 Statistical Summary (Numerical):")
print(df.describe().round(2))

print("\n📊 Statistical Summary (Categorical):")
print(df.describe(include="object"))

# ── 4. MISSING VALUES ────────────────────────────────────────
print("\n🔍 Missing Values:")
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({"Missing Count": missing, "Missing %": missing_pct})
missing_df = missing_df[missing_df["Missing Count"] > 0].sort_values("Missing %", ascending=False)
print(missing_df)

# ── 5. SURVIVAL RATE ─────────────────────────────────────────
print("\n🎯 Overall Survival Rate:")
print(df["survived"].value_counts())
print(f"   Survival Rate: {df['survived'].mean() * 100:.2f}%")

# ── 6. VISUALIZATIONS ────────────────────────────────────────
fig = plt.figure(figsize=(18, 22))
fig.suptitle("Titanic EDA — Key Insights", fontsize=20, fontweight="bold", y=0.98)
gs = GridSpec(4, 3, figure=fig, hspace=0.5, wspace=0.4)

# 6.1 Survival Count
ax1 = fig.add_subplot(gs[0, 0])
df["survived"].value_counts().plot(kind="bar", ax=ax1, color=["#e74c3c", "#2ecc71"], edgecolor="black")
ax1.set_title("Survival Count")
ax1.set_xticklabels(["Did Not Survive", "Survived"], rotation=0)
ax1.set_ylabel("Count")
for p in ax1.patches:
    ax1.annotate(str(p.get_height()), (p.get_x() + p.get_width() / 2., p.get_height()),
                 ha="center", va="bottom", fontsize=10)

# 6.2 Survival by Sex
ax2 = fig.add_subplot(gs[0, 1])
sns.countplot(data=df, x="sex", hue="survived", ax=ax2, palette=["#e74c3c", "#2ecc71"])
ax2.set_title("Survival by Sex")
ax2.set_xlabel("Sex")
ax2.legend(["Did Not Survive", "Survived"])

# 6.3 Survival by Passenger Class
ax3 = fig.add_subplot(gs[0, 2])
sns.countplot(data=df, x="pclass", hue="survived", ax=ax3, palette=["#e74c3c", "#2ecc71"])
ax3.set_title("Survival by Passenger Class")
ax3.set_xlabel("Class")
ax3.legend(["Did Not Survive", "Survived"])

# 6.4 Age Distribution
ax4 = fig.add_subplot(gs[1, 0])
df["age"].dropna().plot(kind="hist", bins=30, ax=ax4, color="#3498db", edgecolor="black", alpha=0.8)
ax4.set_title("Age Distribution")
ax4.set_xlabel("Age")
ax4.set_ylabel("Count")

# 6.5 Age by Survival (KDE)
ax5 = fig.add_subplot(gs[1, 1])
for survived, color, label in [(0, "#e74c3c", "Did Not Survive"), (1, "#2ecc71", "Survived")]:
    df[df["survived"] == survived]["age"].dropna().plot(kind="kde", ax=ax5, color=color, label=label, linewidth=2)
ax5.set_title("Age Distribution by Survival")
ax5.set_xlabel("Age")
ax5.legend()

# 6.6 Fare Distribution (log scale)
ax6 = fig.add_subplot(gs[1, 2])
df["fare"].dropna().plot(kind="hist", bins=40, ax=ax6, color="#9b59b6", edgecolor="black", alpha=0.8)
ax6.set_title("Fare Distribution")
ax6.set_xlabel("Fare (£)")
ax6.set_ylabel("Count")

# 6.7 Survival Rate by Class and Sex (heatmap)
ax7 = fig.add_subplot(gs[2, 0:2])
pivot = df.pivot_table(values="survived", index="sex", columns="pclass", aggfunc="mean")
sns.heatmap(pivot, annot=True, fmt=".2%", cmap="RdYlGn", ax=ax7, linewidths=0.5, linecolor="gray")
ax7.set_title("Survival Rate by Class & Sex")
ax7.set_xlabel("Passenger Class")
ax7.set_ylabel("Sex")

# 6.8 Embarkation Port vs Survival
ax8 = fig.add_subplot(gs[2, 2])
sns.countplot(data=df, x="embarked", hue="survived", ax=ax8, palette=["#e74c3c", "#2ecc71"])
ax8.set_title("Survival by Embarkation Port")
ax8.set_xlabel("Port (C=Cherbourg, Q=Queenstown, S=Southampton)")
ax8.legend(["Did Not Survive", "Survived"])

# 6.9 Correlation Heatmap
ax9 = fig.add_subplot(gs[3, :])
num_cols = ["survived", "pclass", "age", "sibsp", "parch", "fare"]
corr = df[num_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm", ax=ax9,
            linewidths=0.5, vmin=-1, vmax=1, center=0)
ax9.set_title("Correlation Matrix (Numerical Features)")

plt.savefig("eda_titanic_report.png", dpi=150, bbox_inches="tight")
plt.close()
print("\n✅ Visualizations saved to: eda_titanic_report.png")

# ── 7. KEY FINDINGS ──────────────────────────────────────────
print("\n" + "=" * 60)
print("📋 KEY FINDINGS SUMMARY")
print("=" * 60)

total = len(df)
survived = df["survived"].sum()
print(f"\n1. Total Passengers : {total}")
print(f"   Survived         : {survived} ({survived/total*100:.1f}%)")
print(f"   Did Not Survive  : {total - survived} ({(total-survived)/total*100:.1f}%)")

print("\n2. Survival by Sex:")
for sex, grp in df.groupby("sex"):
    rate = grp["survived"].mean() * 100
    print(f"   {sex.capitalize():<8}: {rate:.1f}%")

print("\n3. Survival by Class:")
for cls, grp in df.groupby("pclass"):
    rate = grp["survived"].mean() * 100
    print(f"   Class {cls}: {rate:.1f}%")

print("\n4. Average Age:")
print(f"   Overall   : {df['age'].mean():.1f} years")
print(f"   Survivors : {df[df['survived']==1]['age'].mean():.1f} years")
print(f"   Non-Surv. : {df[df['survived']==0]['age'].mean():.1f} years")

print("\n5. Average Fare:")
print(f"   Survivors : £{df[df['survived']==1]['fare'].mean():.2f}")
print(f"   Non-Surv. : £{df[df['survived']==0]['fare'].mean():.2f}")

print("\n6. Strongest Correlations with Survival:")
corr_survived = df[num_cols].corr()["survived"].drop("survived").abs().sort_values(ascending=False)
for feat, val in corr_survived.items():
    print(f"   {feat:<10}: {val:.3f}")

print("\n✅ EDA Complete!")


TITANIC DATASET — EXPLORATORY DATA ANALYSIS

📌 Shape: (891, 15)

📌 Column Names:
 ['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked', 'class', 'who', 'adult_male', 'deck', 'embark_town', 'alive', 'alone']

📌 Data Types:
 survived          int64
pclass            int64
sex              object
age             float64
sibsp             int64
parch             int64
fare            float64
embarked         object
class          category
who              object
adult_male         bool
deck           category
embark_town      object
alive            object
alone              bool
dtype: object

📌 First 5 Rows:
    survived  pclass     sex   age  sibsp  parch     fare embarked  class  \
0         0       3    male  22.0      1      0   7.2500        S  Third   
1         1       1  female  38.0      1      0  71.2833        C  First   
2         1       3  female  26.0      0      0   7.9250        S  Third   
3         1       1  female  35.0      1      0  53.1000    